In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, re
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import time
import PINN
from PINN import reader, models, pl, tl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchdiffeq import odeint

import matplotlib as mpl
import seaborn as sns
from matplotlib.patches import Patch

import palantir
# import cellrank as cr
import scvelo as scv


os.chdir("/home/wz369/rds/hpc-work/PINN_dynamics")

In [2]:
sc.settings.set_figure_params(frameon=False, dpi=50, figsize=(3,3))

In [3]:
adata = sc.read_h5ad('data/ery_mk_Aug1.h5ad')

# load trajectory

In [32]:
def read_trajectory(result_dir):
    """
    read trajectory npy into dict
    """
    trajectory_dict = {}

    npy_save_dir = os.path.join(result_dir, 'Density_transport')

    for files in os.listdir(npy_save_dir):

        day = files.split('_')[1].replace('Day', '')

        if files.endswith("sim_trajectory.npy"):
            trajectory_dict[day] = np.load(os.path.join(npy_save_dir, files))

    return trajectory_dict             

traj_dict_V0 = read_trajectory("results/ery_mk_Aug1_multiscaled/pde_params_tsense_0")
traj_dict_V1 = read_trajectory("results/ery_mk_Aug1_multiscaled/pde_params_tsense_1")


# evaluate by distance

In [33]:
cbs_t = [ adata.obs.query("`timepoint_tx_days` == @t & `anno_man` == 'HSC'").index.tolist()
                for t in adata.uns['pop']['t'] 
         ] 

for i, t in enumerate(adata.uns['pop']['t'][:-1]):

    print(t, len(cbs_t[i]), traj_dict_V0[str(t)].shape)

3 105 (11, 105, 3)
7 130 (11, 130, 3)
12 67 (11, 67, 3)
27 386 (11, 386, 3)
49 281 (11, 281, 3)
76 498 (11, 498, 3)
112 704 (11, 704, 3)
161 518 (11, 518, 3)


In [15]:
from torchcfm.optimal_transport import wasserstein

In [49]:
adata

AnnData object with n_obs × n_vars = 22740 × 4814
    obs: 'n_genes', 'n_counts', 'mt_count', 'mt_frac', 'doublet_scores', 'predicted_doublets', 'xist_logn', 'Ygene_logn', 'xist_bin', 'Ygene_bin', 'sex_adata', 'biosample_id', 'cellid', 'RBG', 'SLXid', 'index', '10xsample_description', 'sex_mixed', 'sex_meta', 'mouse_id', 'sortedcells', 'expected_cells_10x', 'cellranger_cellsfound', 'chemistry', 'tom', 'expdate', 'batch', 'timepoint_tx_days', 'start_age', 'sample_id', 'countfile', 'S_score', 'G2M_score', 'phase', 'leiden', 'SLX', 'plate_sorted', 'plate_rearranged', 'well_sorted', 'well_rearranged', 'set_index', 'CI_index', 'mouse_platelabel', 'sort_method', 'sample.name', 'population', 'sex', 'countfolder', 'batch_plate_sorted', 'data_type', 'sex_combined', 'longname', 'anno_man', 'leiden_DM', 'HSCscore', 'nn_HSCscore', 'isroot', 'dpt_pseudotime', 'leiden_orig', 'logk', 'net_prolif', 'log10SR', 'log_density_at_E3', 'log_density_at_E7', 'log_density_at_E12', 'log_density_at_E12_clip', 'l

In [50]:
w2_dict = {}

for t, tra in traj_dict_V0.items():

    X_t = adata[adata.obs.timepoint_tx_days == int(t)].obsm['DM_EigenVectors_multiscaled'].copy()

    w2_list = []
    for i in range(11):
        w2 = wasserstein(torch.tensor(tra[i]), 
                torch.tensor(X_t), 
                power=2, method='exact')
        
        w2_list.append(w2)  
    
    w2_dict[t] = w2_list

In [51]:
w2_dict['3']

[1.1231226177207443,
 1.0927678334008089,
 1.070612380840337,
 1.0547793534762244,
 1.0442695038634524,
 1.0386761850967106,
 1.0379167523535335,
 1.0420264193139843,
 1.0510395588796215,
 1.0649471700716096,
 1.0836546533237605]

In [ ]:
w2_dict['12'] # ??!!

[2.700648552498363,
 2.7486288106076313,
 2.8459317786345513,
 2.9839147355876743,
 3.1555940730159815,
 3.3554669763606517,
 3.5790361736122533,
 3.8224594610676768,
 4.0824704283414865,
 4.356340633735738,
 4.642157898396127]

In [55]:
w2_dict['49'] # ??!!

[3.6623651313364296,
 4.103891701288019,
 4.600531995258613,
 5.138162380874716,
 5.706599983838586,
 6.298369378416872,
 6.907839375404278,
 7.53063033829504,
 8.163516832936878,
 8.80434263110763,
 9.45148750207127]